[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mongodb-partners/partner-cookbooks/blob/main/partners/langchain/advanced-rag/02_fulltext_hybrid_and_reranking.ipynb)

# Advanced RAG on MongoDB Atlas: 2. Full-Text, Hybrid & Reranking

Notebook 1 covered semantic (vector) search. Vector search is great at *meaning* but can miss
exact terms like names, places, or rare keywords. This notebook adds the retrieval techniques that
fix that:

## What you'll learn
1. **Full-text search**: classic keyword (BM25) search with an Atlas Search index.
2. **Hybrid search**: combine vector and full-text results with MongoDB's native `$rankFusion`.
3. **Reranking**: reorder candidates with a cross-encoder for sharper relevance.
4. Compare all four retrievers on the same query and see where each wins.

## Prerequisites
Finish `01_atlas_setup_and_vector_search.ipynb` first. It creates the `movies_vector_index`
this notebook reuses. Same credentials as before.


## Setup

In [1]:
%%capture
%pip install -q \
    langchain==1.4.5 langchain-mongodb==0.12.1 langchain-anthropic==1.7.6 \
    pymongo==4.18.3 python-dotenv==1.2.4 requests==2.34.2

In [2]:
# Fetch the shared helper if it isn't already beside the notebook (Colab).
import os, urllib.request

if not os.path.exists("utils.py"):
    url = ("https://raw.githubusercontent.com/mongodb-partners/partner-cookbooks"
           "/main/partners/langchain/advanced-rag/utils.py")
    urllib.request.urlretrieve(url, "utils.py")
    print("Downloaded utils.py")

In [3]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from utils import AtlasEmbeddings, connect, get_secret, setting

MONGODB_URI  = get_secret("MONGODB_URI",  "MongoDB Atlas connection string: ")
ATLAS_AI_KEY = get_secret("MDB_ATLAS_API_KEY", "Atlas API key (for embeddings/rerank): ")
EMBEDDING_MODEL = setting("EMBEDDING_MODEL", "voyage-3-large")
EMBEDDING_DIMS  = int(setting("EMBEDDING_DIMENSIONS", "2048"))
DB_NAME         = setting("MONGODB_DB", "sample_mflix")

client = connect(MONGODB_URI)
# `db` as well as the collection: the index-budget guard below inspects the whole database.
db     = client[DB_NAME]
movies = db["embedded_movies"]
embeddings = AtlasEmbeddings(api_key=ATLAS_AI_KEY, model=EMBEDDING_MODEL,
                             dimensions=EMBEDDING_DIMS)

# Both notebooks 1 and 2 use these two index names, and each creates whichever is
# missing, so the two can run in either order (or independently).
VECTOR_INDEX   = "movies_vector_index"
FULLTEXT_INDEX = "movies_fulltext_index"
print("Ready.")

Ready.


## 1. Full-text (keyword) search

A **full-text search index** lets Atlas rank documents by keyword relevance (the BM25 algorithm),
just like a search engine. We index the `plot` field.


In [4]:
from langchain_mongodb.index import create_fulltext_search_index
from utils import assert_room_for_index

if FULLTEXT_INDEX not in {ix["name"] for ix in movies.list_search_indexes()}:
    # At the search index limit this raises and names what is occupying the slots,
    # rather than removing one to make room.
    assert_room_for_index(db, FULLTEXT_INDEX)
    print("Creating full-text index, waiting until queryable...")
    create_fulltext_search_index(
        collection=movies, index_name=FULLTEXT_INDEX,
        field="plot", wait_until_complete=420,
    )
    print("Index is ready.")
else:
    print("Index already exists.")

Index already exists.


In [5]:
from langchain_mongodb.retrievers import MongoDBAtlasFullTextSearchRetriever

fulltext_retriever = MongoDBAtlasFullTextSearchRetriever(
    collection=movies, search_index_name=FULLTEXT_INDEX,
    search_field="plot", k=5,
)

print("Full-text results for 'shark attack in the ocean':")
for d in fulltext_retriever.invoke("shark attack in the ocean"):
    print(f"  - {d.metadata['title']}")

Full-text results for 'shark attack in the ocean':
  - Deep Rising
  - How I Ended This Summer
  - The Poseidon Adventure
  - Ewoks: The Battle for Endor
  - Rebellion


Full-text search matches the *words* you typed. That's exactly right when the query contains
specific terms, but it can't tell that "a great white terrorizes swimmers" is about the same thing
as "shark attack." Vector search handles that; full-text handles exact terms. **Hybrid search
gives you both.**

## 2. Hybrid search with `$rankFusion`

MongoDB 8.0 introduced `$rankFusion`, which runs a vector search and a full-text search and fuses
their rankings server-side using Reciprocal Rank Fusion, in one query with no client-side merging.
`MongoDBAtlasHybridSearchRetriever` wraps it: give it the vector store from notebook 1 and the
full-text index name.


In [6]:
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_mongodb.retrievers import MongoDBAtlasHybridSearchRetriever

vector_store = MongoDBAtlasVectorSearch(
    collection=movies, embedding=embeddings, index_name=VECTOR_INDEX,
    text_key="plot", embedding_key="plot_embedding_voyage_3_large",
    relevance_score_fn="cosine",
)

hybrid_retriever = MongoDBAtlasHybridSearchRetriever(
    vectorstore=vector_store, search_index_name=FULLTEXT_INDEX, k=5,
)

print("Hybrid results for 'shark attack in the ocean':")
for d in hybrid_retriever.invoke("shark attack in the ocean"):
    print(f"  - {d.metadata['title']}")

Hybrid results for 'shark attack in the ocean':


  - Deep Rising
  - Deep Blue Sea
  - How I Ended This Summer
  - Deep Blue Sea
  - The Poseidon Adventure


The hybrid results are noticeably more on-topic: the semantic side pulls in the right *kind* of
movie while the keyword side keeps exact matches. You can tune the balance with `vector_weight`
and `fulltext_weight` if one signal should count more than the other.

> **Why does "Deep Blue Sea" appear twice?** The sample dataset really does contain two identical
> "Deep Blue Sea" documents with different `_id`s. Retrieval returns *documents*, not titles, so
> both copies rank. Real-world corpora have duplicates too; the next cell dedupes before reranking.

## 3. Reranking for sharper relevance

Fusion is fast but coarse. A **reranker** is a model that scores each candidate against the query
directly, so it can reorder a shortlist much more precisely. The pattern: retrieve a wider pool
(say 10), rerank, keep the best few. `atlas_rerank` (in `utils.py`) calls MongoDB's hosted
`rerank-2.5` model.

In [7]:
from utils import atlas_rerank

query = "shark attack in the ocean"
pool = hybrid_retriever.invoke(query)  # returns k=5 candidates; raise k for a bigger pool

# Dedupe by title (sample_mflix ships a duplicate "Deep Blue Sea" document).
seen, candidates = set(), []
for d in pool:
    if d.metadata["title"] not in seen:
        seen.add(d.metadata["title"])
        candidates.append(d)

titles_by_plot = {d.page_content: d.metadata["title"] for d in candidates}

print("Before rerank:")
for d in candidates:
    print(f"  - {d.metadata['title']}")

reranked_plots = atlas_rerank(query, [d.page_content for d in candidates],
                              api_key=ATLAS_AI_KEY, top_n=3)
print("\nAfter rerank (top 3):")
for plot in reranked_plots:
    print(f"  - {titles_by_plot[plot]}")

Before rerank:
  - Deep Rising
  - Deep Blue Sea
  - How I Ended This Summer
  - The Poseidon Adventure



After rerank (top 3):
  - Deep Blue Sea
  - Deep Rising
  - The Poseidon Adventure


## 4. Side-by-side comparison

The same query through each retriever makes the trade-offs concrete.


In [8]:
query = "a giant creature from the deep sea attacks a ship"

def titles(docs):
    return [d.metadata["title"] for d in docs]

vec = titles(vector_store.similarity_search(query, k=5))
ft  = titles(fulltext_retriever.invoke(query))
hyb = titles(hybrid_retriever.invoke(query))

print(f"{'Vector':<28}{'Full-text':<28}{'Hybrid':<28}")
print("-" * 84)
for i in range(5):
    row = [lst[i] if i < len(lst) else "" for lst in (vec, ft, hyb)]
    print(f"{row[0][:26]:<28}{row[1][:26]:<28}{row[2][:26]:<28}")

Vector                      Full-text                   Hybrid                      
------------------------------------------------------------------------------------
Deep Rising                 The Water Horse             The Water Horse             
Pacific Rim                 Sky Captain and the World   Deep Rising                 
Dead Sea                    Big Ass Spider!             Sky Captain and the World   
20,000 Leagues Under the S  Mermaid Chronicles Part 1:  Pacific Rim                 
Dagon                       Carny                       Big Ass Spider!             


## Housekeeping

**Created by this notebook:** `movies_fulltext_index` on `embedded_movies`, if it was not already
there, plus `movies_vector_index` if notebook 1 has not run -- so this notebook works on its own.

**Deleted by this notebook:** nothing. It reads `sample_mflix` and writes no documents to it, and it
drops no collection and no index.

**Left in place, deliberately:** both indexes. Notebook 3 needs the slot arithmetic to work out, and
the Mastra set in this repository uses the same two names. Sharing them is what lets eight notebooks
fit in a free cluster's three index slots.

**At the search index limit** the creation cell stops and lists what is using the slots rather than
freeing one.

**To remove what this notebook added**, drop `movies_fulltext_index` from the Atlas UI, once you are
done with notebook 3 and the Mastra set.

## Recap

You built keyword (full-text) search, combined it with vector search via native `$rankFusion`
hybrid retrieval, and sharpened the ranking with a reranker: the core toolkit of modern retrieval.

**Next:** `03_graphlookup_and_auto_embedding.ipynb` explores two more MongoDB capabilities:
traversing relationships with `$graphLookup` (a movie knowledge graph) and letting Atlas generate
embeddings automatically at write time.
